# Evaluation: Fine-Tuned Mistral 7B vs GPT-4o-mini

Compare fine-tuned model (checkpoint-30, eval_loss=0.5480) against GPT-4o-mini baseline
on the same eval set. Measures per-field extraction accuracy.

### 1. Environment Setup

In [ ]:
!pip install -q peft trl bitsandbytes wandb sentencepiece pydantic rapidfuzz openai python-dotenv

import os, sys, json, torch, gc
from tqdm import tqdm

In [ ]:
# Check GPU
gpu_name = !nvidia-smi --query-gpu=name --format=csv,noheader
print(f'GPU: {gpu_name[0]}')
if 'P100' in gpu_name[0]:
    raise RuntimeError('P100 (sm_60) not compatible with bitsandbytes. Restart for T4.')

### 2. API Keys

In [ ]:
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN'
os.environ['AZURE_OPENAI_API_KEY'] = 'YOUR_AZURE_KEY'
os.environ['AZURE_OPENAI_ENDPOINT'] = 'YOUR_AZURE_ENDPOINT'
os.environ['AZURE_OPENAI_DEPLOYMENT'] = 'gpt-4o-mini'
os.environ['AZURE_OPENAI_API_VERSION'] = '2024-12-01-preview'

### 3. Clone Code

In [ ]:
!git clone https://github.com/Pushparaj13811/invoice-extraction-mistral-7b-fine-tuning.git
sys.path.insert(0, '/kaggle/working/invoice-extraction-mistral-7b-fine-tuning')

### 4. Find Adapter & Eval Data

The adapter is from checkpoint-30 (best checkpoint, eval_loss=0.5480).
Uploaded as Kaggle dataset `mistral-7b-invoice-adapter-v2`.

In [ ]:
import subprocess

result = subprocess.run(['find', '/kaggle/input', '-name', 'adapter_config.json'], capture_output=True, text=True)
ADAPTER_PATH = os.path.dirname(result.stdout.strip().split('\n')[0])
print(f'Adapter: {ADAPTER_PATH}')
print(f'Files: {os.listdir(ADAPTER_PATH)}')

In [ ]:
result = subprocess.run(['find', '/kaggle/input', '-name', 'eval.jsonl'], capture_output=True, text=True)
EVAL_PATH = result.stdout.strip().split('\n')[0]
print(f'Eval: {EVAL_PATH}')

### 5. Load Eval Data

In [ ]:
from src.data.format import load_jsonl
from src.data.schema import Invoice

eval_data = load_jsonl(EVAL_PATH)
gold_invoices = [Invoice.model_validate_json(ex['output']) for ex in eval_data]
print(f'Eval samples: {len(eval_data)}')

### 6. Load Fine-Tuned Model

In [ ]:
from src.evaluation.evaluate import load_finetuned_model

model, tokenizer = load_finetuned_model('mistralai/Mistral-7B-v0.3', ADAPTER_PATH)
model.eval()
print('Model loaded')

### 7. Sanity Check

Quick test — does the model produce valid JSON?

In [ ]:
test_input = """Invoice #TEST-001
From: Acme Corp, 123 Main St, New York, NY 10001
Date: 2024-06-15
Due: 2024-07-15

Items:
Widget A x2 @ $50.00 = $100.00
Widget B x1 @ $25.00 = $25.00

Subtotal: $125.00
Tax: $10.00
Total: $135.00
Currency: USD
Payment Terms: Net 30
"""

prompt = f"""### Instruction:
Extract all invoice fields from the following invoice text as JSON.

### Input:
{test_input}

### Response:
"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=600,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    out[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print("=== Model Output ===")
print(response[:500])

### 8. Run Fine-Tuned Model Inference

In [ ]:
from src.evaluation.evaluate import run_finetuned_inference

ft_predictions = run_finetuned_inference(model, tokenizer, eval_data, max_new_tokens=600)

ft_parsed = sum(1 for p in ft_predictions if p is not None)
print(f'Parsed: {ft_parsed}/{len(ft_predictions)} ({ft_parsed/len(ft_predictions)*100:.1f}%)')

### 9. Cleanup GPU Memory

In [ ]:
del model, tokenizer
gc.collect()
torch.cuda.empty_cache()
print('GPU freed')

### 10. Run GPT-4o-mini Baseline

Zero-shot extraction using the same schema in the system prompt.

In [ ]:
from src.evaluation.baseline import run_baseline

baseline_predictions = run_baseline(eval_data)

bl_parsed = sum(1 for p in baseline_predictions if p is not None)
print(f'Parsed: {bl_parsed}/{len(baseline_predictions)} ({bl_parsed/len(baseline_predictions)*100:.1f}%)')

### 11. Compute Metrics

In [ ]:
from src.evaluation.evaluate import aggregate_metrics

ft_metrics = aggregate_metrics(ft_predictions, gold_invoices)
baseline_metrics = aggregate_metrics(baseline_predictions, gold_invoices)

### 12. Model Comparison

In [ ]:
print(f"{'Metric':<25} {'Fine-Tuned':>12} {'GPT-4o-mini':>12}")
print('-' * 52)
print(f"{'Overall Accuracy':<25} {ft_metrics['overall_accuracy']:>11.1%} {baseline_metrics['overall_accuracy']:>11.1%}")
print(f"{'JSON Parse Rate':<25} {ft_metrics['json_parse_success_rate']:>11.1%} {baseline_metrics['json_parse_success_rate']:>11.1%}")
print(f"{'Line Item Score':<25} {ft_metrics['line_item_score']:>11.1%} {baseline_metrics['line_item_score']:>11.1%}")

imp = ((ft_metrics['overall_accuracy'] - baseline_metrics['overall_accuracy']) / baseline_metrics['overall_accuracy'] * 100) if baseline_metrics['overall_accuracy'] > 0 else 0
print(f'\nImprovement: {imp:+.1f}%')

### 13. Per-Field Accuracy

In [ ]:
print(f"{'Field':<25} {'Fine-Tuned':>12} {'GPT-4o-mini':>12} {'Improvement':>12}")
print('-' * 64)
for field in sorted(set(list(ft_metrics['per_field'].keys()) + list(baseline_metrics['per_field'].keys()))):
    ft_val = ft_metrics['per_field'].get(field, 0)
    bl_val = baseline_metrics['per_field'].get(field, 0)
    imp = ((ft_val - bl_val) / bl_val * 100) if bl_val > 0 else 0
    print(f'{field:<25} {ft_val:>11.1%} {bl_val:>11.1%} {imp:>+11.1f}%')

### 14. Save Report

In [ ]:
from src.evaluation.evaluate import generate_report

report = generate_report(ft_metrics, baseline_metrics)
print(report)

In [ ]:
with open('/kaggle/working/evaluation_report.md', 'w') as f:
    f.write(report)

results = {
    'ft_metrics': ft_metrics,
    'baseline_metrics': baseline_metrics,
    'ft_predictions': [p.model_dump() if p else None for p in ft_predictions],
    'baseline_predictions': [p.model_dump() if p else None for p in baseline_predictions],
}
with open('/kaggle/working/evaluation_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('Reports saved')